# Notebook 03 — Create SQL Database

## Objective

This notebook creates a lightweight SQLite database from the dashboard-ready CSV tables generated in previous notebooks.

The database is designed as a reporting layer for the Power BI dashboard.

## Inputs

CSV tables from:

- `data/dashboard_exports/dim_slice.csv`
- `data/dashboard_exports/fact_slice_features.csv`
- `data/dashboard_exports/fact_anomaly_results.csv`
- `data/dashboard_exports/summary_anomaly_categories.csv`
- `data/dashboard_exports/summary_phase_fractions.csv`
- `data/dashboard_exports/summary_2d_3d_objects.csv`
- `data/dashboard_exports/image_index.csv`

## Outputs

- `data/sqlite/microct_quality_monitoring.db`
- `sql/create_tables.sql`
- `sql/example_queries.sql`

The SQLite database provides a clean data model that can be connected to Power BI or queried directly using SQL.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

# Project paths
PROJECT_ROOT = Path("..").resolve()

DASHBOARD_EXPORTS = PROJECT_ROOT / "data" / "dashboard_exports"
SQLITE_DIR = PROJECT_ROOT / "data" / "sqlite"
SQL_DIR = PROJECT_ROOT / "sql"

SQLITE_DIR.mkdir(parents=True, exist_ok=True)
SQL_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = SQLITE_DIR / "microct_quality_monitoring.db"

print("Project 05 root:", PROJECT_ROOT)
print("Dashboard exports:", DASHBOARD_EXPORTS)
print("SQLite directory:", SQLITE_DIR)
print("SQL directory:", SQL_DIR)
print("Database path:", DB_PATH)

Project 05 root: C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi
Dashboard exports: C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\dashboard_exports
SQLite directory: C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\sqlite
SQL directory: C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\sql
Database path: C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\sqlite\microct_quality_monitoring.db


## 1. Inspect dashboard-ready CSV files

Before creating the SQLite database, we verify that all expected CSV files are available.

In [2]:
# Expected CSV files from previous notebooks

expected_csv_files = {
    "dim_slice": DASHBOARD_EXPORTS / "dim_slice.csv",
    "fact_slice_features": DASHBOARD_EXPORTS / "fact_slice_features.csv",
    "fact_anomaly_results": DASHBOARD_EXPORTS / "fact_anomaly_results.csv",
    "summary_anomaly_categories": DASHBOARD_EXPORTS / "summary_anomaly_categories.csv",
    "summary_phase_fractions": DASHBOARD_EXPORTS / "summary_phase_fractions.csv",
    "summary_2d_3d_objects": DASHBOARD_EXPORTS / "summary_2d_3d_objects.csv",
    "image_index": DASHBOARD_EXPORTS / "image_index.csv",
}

for table_name, path in expected_csv_files.items():
    print(f"{table_name:30s} | exists: {path.exists()} | {path}")

dim_slice                      | exists: True | C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\dashboard_exports\dim_slice.csv
fact_slice_features            | exists: True | C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\dashboard_exports\fact_slice_features.csv
fact_anomaly_results           | exists: True | C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\dashboard_exports\fact_anomaly_results.csv
summary_anomaly_categories     | exists: True | C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\dashboard_exports\summary_anomaly_categories.csv
summary_phase_fractions        | exists: True | C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\dashboard_exports\summary_phase_fractions.csv
summary_2d_3d_objects          | exists: True | C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_qual

In [3]:
# Load CSV files into pandas DataFrames

tables = {}

for table_name, path in expected_csv_files.items():
    tables[table_name] = pd.read_csv(path)
    print(f"{table_name:30s} | shape: {tables[table_name].shape}")

display(tables["dim_slice"].head())
display(tables["fact_slice_features"].head())
display(tables["fact_anomaly_results"].head())

dim_slice                      | shape: (435, 5)
fact_slice_features            | shape: (435, 23)
fact_anomaly_results           | shape: (435, 9)
summary_anomaly_categories     | shape: (8, 8)
summary_phase_fractions        | shape: (3, 11)
summary_2d_3d_objects          | shape: (2, 23)
image_index                    | shape: (104, 16)


,slice_id,z_um,z_mm,relative_depth_percent,slice_zone
0,0,0.0,0.00,0.000000,bottom_0_25
1,1,80.0,0.08,0.230415,bottom_0_25
2,2,160.0,0.16,0.460829,bottom_0_25
3,3,240.0,0.24,0.691244,bottom_0_25
4,4,320.0,0.32,0.921659,bottom_0_25


,slice_id,pore_fraction,resin_fraction,matrix_fraction,assigned_fraction,mean_intensity,std_intensity,mean_intensity_pores,mean_intensity_resin,mean_intensity_matrix,pore_object_count,mean_pore_area_um2,median_pore_area_um2,mean_pore_equivalent_diameter_um,median_pore_equivalent_diameter_um,resin_object_count,mean_resin_area_um2,median_resin_area_um2,mean_resin_equivalent_diameter_um,median_resin_equivalent_diameter_um,heterogeneity_index,phase_fraction_sum,unassigned_fraction
0,0,0.056875,0.303065,0.622066,0.982006,162.006452,34.223109,76.737897,136.006699,182.515447,250,52352.000000,19200.0,216.142166,156.352804,592,117805.405405,12800.0,233.533527,127.66153,0.231376,0.982006,0.017994
1,1,0.049366,0.302342,0.638364,0.990071,163.499583,33.344875,77.197183,136.707571,182.891997,251,45258.964143,12800.0,200.100077,127.661530,769,90473.862159,12800.0,216.507405,127.66153,0.241253,0.990071,0.009929
2,2,0.040800,0.294249,0.657276,0.992324,165.032206,32.662106,77.162236,136.070321,183.481699,261,35972.413793,12800.0,179.854408,127.661530,755,89684.768212,12800.0,221.468784,127.66153,0.252997,0.992324,0.007676
3,3,0.033513,0.284570,0.673963,0.992046,166.618951,32.181359,77.149378,135.372068,184.301490,249,30971.887550,12800.0,171.086291,127.661530,704,93018.181818,12800.0,225.478737,127.66153,0.263488,0.992046,0.007954
4,4,0.028952,0.265491,0.697630,0.992074,168.204917,32.126347,77.017291,133.324743,185.328058,260,25624.615385,12800.0,155.146942,127.661530,672,90914.285714,12800.0,234.745590,127.66153,0.276853,0.992074,0.007926


,slice_id,PC1,PC2,is_anomaly,anomaly_score,anomaly_category,anomaly_flag,anomaly_status,anomaly_score_abs
0,0,-16.003458,-2.397432,True,0.232357,high_porosity+resin_rich+resin_fragmented+larg...,1,Anomalous,0.232357
1,1,-16.159629,-4.010913,True,0.224484,high_porosity+resin_rich+resin_fragmented+larg...,1,Anomalous,0.224484
2,2,-14.789948,-4.687371,True,0.208778,high_porosity+resin_rich+resin_fragmented+larg...,1,Anomalous,0.208778
3,3,-13.171530,-4.773668,True,0.180866,high_porosity+resin_rich+resin_fragmented,1,Anomalous,0.180866
4,4,-11.460013,-4.886507,True,0.164346,high_porosity+resin_rich+resin_fragmented,1,Anomalous,0.164346


## 2. Validate table consistency

This section performs basic consistency checks before writing the SQLite database.

The main validation is that all slice-level tables share the same `slice_id` domain.

In [4]:
# Validate slice_id consistency across slice-level tables

slice_ids_dim = set(tables["dim_slice"]["slice_id"])
slice_ids_features = set(tables["fact_slice_features"]["slice_id"])
slice_ids_anomaly = set(tables["fact_anomaly_results"]["slice_id"])

print("Number of slice IDs in dim_slice:", len(slice_ids_dim))
print("Number of slice IDs in fact_slice_features:", len(slice_ids_features))
print("Number of slice IDs in fact_anomaly_results:", len(slice_ids_anomaly))

print("\nDim vs features match:", slice_ids_dim == slice_ids_features)
print("Dim vs anomaly match:", slice_ids_dim == slice_ids_anomaly)

# Image index only contains selected slices, so it should be a subset of dim_slice
slice_ids_images = set(tables["image_index"]["slice_id"])

print("\nNumber of slice IDs in image_index:", len(slice_ids_images))
print("Image slice IDs are subset of dim_slice:", slice_ids_images.issubset(slice_ids_dim))

# Check duplicate keys where tables should be unique by slice_id
print("\nDuplicate checks:")
print("dim_slice duplicated slice_id:", tables["dim_slice"]["slice_id"].duplicated().sum())
print("fact_slice_features duplicated slice_id:", tables["fact_slice_features"]["slice_id"].duplicated().sum())
print("fact_anomaly_results duplicated slice_id:", tables["fact_anomaly_results"]["slice_id"].duplicated().sum())

# image_index should have multiple records per slice because it stores different image types
print("image_index duplicated slice_id:", tables["image_index"]["slice_id"].duplicated().sum())

Number of slice IDs in dim_slice: 435
Number of slice IDs in fact_slice_features: 435
Number of slice IDs in fact_anomaly_results: 435

Dim vs features match: True
Dim vs anomaly match: True

Number of slice IDs in image_index: 26
Image slice IDs are subset of dim_slice: True

Duplicate checks:
dim_slice duplicated slice_id: 0
fact_slice_features duplicated slice_id: 0
fact_anomaly_results duplicated slice_id: 0
image_index duplicated slice_id: 78


## 3. Write tables to SQLite database

This section writes all dashboard-ready tables into a SQLite database.

Each CSV file becomes one SQL table with the same name.

In [5]:
# Create SQLite database and write tables

# Remove existing database if it exists, to avoid stale tables during development
if DB_PATH.exists():
    DB_PATH.unlink()
    print("Existing database removed:", DB_PATH)

with sqlite3.connect(DB_PATH) as conn:
    for table_name, df in tables.items():
        df.to_sql(
            name=table_name,
            con=conn,
            if_exists="replace",
            index=False
        )
        print(f"Written table: {table_name:30s} | shape: {df.shape}")

print("\nSQLite database created:")
print(DB_PATH)

Written table: dim_slice                      | shape: (435, 5)
Written table: fact_slice_features            | shape: (435, 23)
Written table: fact_anomaly_results           | shape: (435, 9)
Written table: summary_anomaly_categories     | shape: (8, 8)
Written table: summary_phase_fractions        | shape: (3, 11)
Written table: summary_2d_3d_objects          | shape: (2, 23)
Written table: image_index                    | shape: (104, 16)

SQLite database created:
C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\sqlite\microct_quality_monitoring.db


In [6]:
# Verify tables stored in SQLite database

with sqlite3.connect(DB_PATH) as conn:
    sql_tables = pd.read_sql_query(
        """
        SELECT name 
        FROM sqlite_master 
        WHERE type = 'table'
        ORDER BY name;
        """,
        conn
    )

display(sql_tables)

with sqlite3.connect(DB_PATH) as conn:
    for table_name in sql_tables["name"]:
        row_count = pd.read_sql_query(
            f"SELECT COUNT(*) AS n_rows FROM {table_name};",
            conn
        )["n_rows"].iloc[0]
        print(f"{table_name:30s} | rows: {row_count}")

,name
0,dim_slice
1,fact_anomaly_results
2,fact_slice_features
3,image_index
4,summary_2d_3d_objects
5,summary_anomaly_categories
6,summary_phase_fractions


dim_slice                      | rows: 435
fact_anomaly_results           | rows: 435
fact_slice_features            | rows: 435
image_index                    | rows: 104
summary_2d_3d_objects          | rows: 2
summary_anomaly_categories     | rows: 8
summary_phase_fractions        | rows: 3


## 4. Run example SQL queries

This section runs example SQL queries to verify that the database can be used as a reporting layer.

The queries are designed to support Power BI dashboard logic and portfolio documentation.

In [7]:
# Example SQL queries for dashboard reporting

example_queries = {
    "phase_fraction_by_zone": """
        SELECT 
            d.slice_zone,
            COUNT(*) AS n_slices,
            AVG(f.pore_fraction) AS mean_pore_fraction,
            AVG(f.resin_fraction) AS mean_resin_fraction,
            AVG(f.matrix_fraction) AS mean_matrix_fraction,
            AVG(f.heterogeneity_index) AS mean_heterogeneity_index
        FROM fact_slice_features f
        JOIN dim_slice d
            ON f.slice_id = d.slice_id
        GROUP BY d.slice_zone
        ORDER BY MIN(d.relative_depth_percent);
    """,

    "anomaly_summary_by_category": """
        SELECT
            a.anomaly_category,
            a.anomaly_status,
            COUNT(*) AS n_slices,
            ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_anomaly_results), 2) AS percentage_slices,
            AVG(a.anomaly_score) AS mean_anomaly_score,
            AVG(f.pore_fraction) AS mean_pore_fraction,
            AVG(f.resin_fraction) AS mean_resin_fraction,
            AVG(f.heterogeneity_index) AS mean_heterogeneity_index
        FROM fact_anomaly_results a
        JOIN fact_slice_features f
            ON a.slice_id = f.slice_id
        GROUP BY a.anomaly_category, a.anomaly_status
        ORDER BY n_slices DESC;
    """,

    "top_anomalous_slices": """
        SELECT
            d.slice_id,
            d.z_mm,
            d.relative_depth_percent,
            a.anomaly_score,
            a.anomaly_category,
            f.pore_fraction,
            f.resin_fraction,
            f.pore_object_count,
            f.resin_object_count,
            f.heterogeneity_index
        FROM fact_anomaly_results a
        JOIN dim_slice d
            ON a.slice_id = d.slice_id
        JOIN fact_slice_features f
            ON a.slice_id = f.slice_id
        WHERE a.anomaly_flag = 1
        ORDER BY ABS(a.anomaly_score) DESC
        LIMIT 10;
    """,

    "image_index_overview": """
        SELECT
            image_type,
            COUNT(*) AS n_images,
            COUNT(DISTINCT slice_id) AS n_unique_slices
        FROM image_index
        GROUP BY image_type
        ORDER BY image_type;
    """,

    "object_analysis_2d_vs_3d": """
        SELECT
            phase,
            objects_2d,
            objects_3d,
            objects_2d_per_3d_object,
            mean_equivalent_diameter_2d_um,
            mean_equivalent_diameter_3d_um,
            mean_diameter_ratio_3d_to_2d
        FROM summary_2d_3d_objects
        ORDER BY phase;
    """
}

with sqlite3.connect(DB_PATH) as conn:
    for query_name, query in example_queries.items():
        print("=" * 100)
        print(query_name)
        result = pd.read_sql_query(query, conn)
        display(result)

phase_fraction_by_zone


,slice_zone,n_slices,mean_pore_fraction,mean_resin_fraction,mean_matrix_fraction,mean_heterogeneity_index
0,bottom_0_25,109,0.006976,0.211211,0.776975,0.325943
1,lower_middle_25_50,109,0.017236,0.202726,0.776367,0.323250
2,upper_middle_50_75,108,0.009017,0.196203,0.790813,0.333378
3,top_75_100,109,0.016059,0.208765,0.771402,0.320530


anomaly_summary_by_category


,anomaly_category,anomaly_status,n_slices,percentage_slices,mean_anomaly_score,mean_pore_fraction,mean_resin_fraction,mean_heterogeneity_index
0,normal,Normal,413,94.94,-0.132527,0.011217,0.202967,0.327668
1,high_porosity+resin_rich+resin_fragmented,Anomalous,6,1.38,0.121663,0.025564,0.254529,0.285901
2,high_porosity+resin_fragmented+large_pores,Anomalous,5,1.15,0.007555,0.045150,0.202130,0.301884
3,resin_rich+resin_fragmented,Anomalous,4,0.92,0.043262,0.011159,0.237985,0.306878
4,high_porosity+resin_rich+resin_fragmented+larg...,Anomalous,3,0.69,0.221873,0.049014,0.299885,0.241875
5,high_porosity+large_pores,Anomalous,2,0.46,0.024762,0.046279,0.200509,0.302087
6,high_porosity+resin_rich,Anomalous,1,0.23,0.004544,0.027728,0.235093,0.296363
7,high_porosity+resin_rich+large_pores,Anomalous,1,0.23,0.002389,0.039799,0.213428,0.299454


top_anomalous_slices


,slice_id,z_mm,relative_depth_percent,anomaly_score,anomaly_category,pore_fraction,resin_fraction,pore_object_count,resin_object_count,heterogeneity_index
0,0,0.00,0.000000,0.232357,high_porosity+resin_rich+resin_fragmented+larg...,0.056875,0.303065,250,592,0.231376
1,1,0.08,0.230415,0.224484,high_porosity+resin_rich+resin_fragmented+larg...,0.049366,0.302342,251,769,0.241253
2,2,0.16,0.460829,0.208778,high_porosity+resin_rich+resin_fragmented+larg...,0.040800,0.294249,261,755,0.252997
3,3,0.24,0.691244,0.180866,high_porosity+resin_rich+resin_fragmented,0.033513,0.284570,249,704,0.263488
4,4,0.32,0.921659,0.164346,high_porosity+resin_rich+resin_fragmented,0.028952,0.265491,260,672,0.276853
5,5,0.40,1.152074,0.150086,high_porosity+resin_rich+resin_fragmented,0.025503,0.254450,270,576,0.285732
6,6,0.48,1.382488,0.132312,high_porosity+resin_rich+resin_fragmented,0.022528,0.244605,296,512,0.293024
7,7,0.56,1.612903,0.101083,high_porosity+resin_rich+resin_fragmented,0.017966,0.238708,272,427,0.300025
8,8,0.64,1.843318,0.084839,resin_rich+resin_fragmented,0.013266,0.241629,224,396,0.302521
9,9,0.72,2.073733,0.049300,resin_rich+resin_fragmented,0.011987,0.239932,190,352,0.304921


image_index_overview


,image_type,n_images,n_unique_slices
0,original,26,26
1,pore_mask,26,26
2,resin_mask,26,26
3,segmentation,26,26


object_analysis_2d_vs_3d


,phase,objects_2d,objects_3d,objects_2d_per_3d_object,mean_equivalent_diameter_2d_um,mean_equivalent_diameter_3d_um,mean_diameter_ratio_3d_to_2d
0,pores,5280,232,22.758621,362.344433,570.020004,1.573144
1,resin,86333,1757,49.136596,499.255427,686.088833,1.374224


## 5. Export SQL example queries

The validated SQL queries are exported to `sql/example_queries.sql`.

These queries document how the SQLite database can be used for reporting and dashboard analysis.

In [8]:
# Export example SQL queries to file

example_queries_path = SQL_DIR / "example_queries.sql"

with open(example_queries_path, "w", encoding="utf-8") as f:
    f.write("-- Example SQL queries for the MicroCT Quality Monitoring Dashboard\n")
    f.write("-- Generated from Notebook 03\n\n")

    for query_name, query in example_queries.items():
        f.write("=" * 100 + "\n")
        f.write(f"-- {query_name}\n")
        f.write("=" * 100 + "\n")
        f.write(query.strip())
        f.write("\n\n")

print("Saved SQL example queries:")
print(example_queries_path)

Saved SQL example queries:
C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\sql\example_queries.sql


## 6. Export SQLite schema

The SQLite schema is exported to `sql/create_tables.sql`.

This file documents the database structure used by the Power BI dashboard.

In [9]:
# Export SQLite CREATE TABLE statements

create_tables_path = SQL_DIR / "create_tables.sql"

with sqlite3.connect(DB_PATH) as conn:
    schema_df = pd.read_sql_query(
        """
        SELECT name, sql
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name;
        """,
        conn
    )

with open(create_tables_path, "w", encoding="utf-8") as f:
    f.write("-- SQLite schema for the MicroCT Quality Monitoring Dashboard\n")
    f.write("-- Generated from Notebook 03\n\n")

    for _, row in schema_df.iterrows():
        f.write(f"-- Table: {row['name']}\n")
        f.write(row["sql"])
        f.write(";\n\n")

print("Saved SQLite schema:")
print(create_tables_path)

display(schema_df)

Saved SQLite schema:
C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\sql\create_tables.sql


,name,sql
0,dim_slice,"CREATE TABLE ""dim_slice"" (\n""slice_id"" INTEGER..."
1,fact_anomaly_results,"CREATE TABLE ""fact_anomaly_results"" (\n""slice_..."
2,fact_slice_features,"CREATE TABLE ""fact_slice_features"" (\n""slice_i..."
3,image_index,"CREATE TABLE ""image_index"" (\n""slice_id"" INTEG..."
4,summary_2d_3d_objects,"CREATE TABLE ""summary_2d_3d_objects"" (\n""phase..."
5,summary_anomaly_categories,"CREATE TABLE ""summary_anomaly_categories"" (\n""..."
6,summary_phase_fractions,"CREATE TABLE ""summary_phase_fractions"" (\n""pha..."


In [10]:
# Final verification of database and SQL documentation files

final_outputs = [
    DB_PATH,
    SQL_DIR / "create_tables.sql",
    SQL_DIR / "example_queries.sql",
]

for path in final_outputs:
    print(f"{path.name:35s} | exists: {path.exists()} | size: {path.stat().st_size if path.exists() else 0} bytes")

microct_quality_monitoring.db       | exists: True | size: 200704 bytes
create_tables.sql                   | exists: True | size: 3340 bytes
example_queries.sql                 | exists: True | size: 3542 bytes


## 7. Notebook summary

This notebook created the SQLite reporting database for the MicroCT Quality Monitoring Power BI project.

The workflow used the dashboard-ready CSV tables generated in previous notebooks and stored them in a lightweight SQLite database.

The final database includes:

- `dim_slice`
- `fact_slice_features`
- `fact_anomaly_results`
- `summary_anomaly_categories`
- `summary_phase_fractions`
- `summary_2d_3d_objects`
- `image_index`

The notebook also exported:

- `create_tables.sql`: database schema documentation.
- `example_queries.sql`: example reporting queries.

This database provides a clean SQL-based reporting layer for the Power BI dashboard.

In [11]:
# Final notebook summary

print("SQLite database creation completed successfully.\n")

print("Database:")
print(DB_PATH)

print("\nTables stored in SQLite:")
with sqlite3.connect(DB_PATH) as conn:
    table_summary = pd.read_sql_query(
        """
        SELECT name 
        FROM sqlite_master 
        WHERE type = 'table'
        ORDER BY name;
        """,
        conn
    )

    for table_name in table_summary["name"]:
        n_rows = pd.read_sql_query(
            f"SELECT COUNT(*) AS n_rows FROM {table_name};",
            conn
        )["n_rows"].iloc[0]
        print(f"- {table_name:30s}: {n_rows} rows")

print("\nSQL documentation files:")
for path in [create_tables_path, example_queries_path]:
    print(f"- {path.name}: {path.exists()}")

SQLite database creation completed successfully.

Database:
C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi\data\sqlite\microct_quality_monitoring.db

Tables stored in SQLite:
- dim_slice                     : 435 rows
- fact_anomaly_results          : 435 rows
- fact_slice_features           : 435 rows
- image_index                   : 104 rows
- summary_2d_3d_objects         : 2 rows
- summary_anomaly_categories    : 8 rows
- summary_phase_fractions       : 3 rows

SQL documentation files:
- create_tables.sql: True
- example_queries.sql: True


In [15]:
from pathlib import Path

project_root = Path(
    r"C:\Users\Peter\ai-imaging-portfolio\projects\05_microct_quality_monitoring_powerbi"
)

expected_files = [
    project_root / "README.md",
    project_root / "requirements.txt",
    
    project_root / "notebooks" / "01_prepare_powerbi_tables.ipynb",
    project_root / "notebooks" / "02_export_slice_images_for_dashboard.ipynb",
    project_root / "notebooks" / "03_create_sql_database.ipynb",
    
    project_root / "data" / "dashboard_exports" / "microct_powerbi_input.xlsx",
    project_root / "data" / "dashboard_exports" / "microct_powerbi_input_corrected.xlsx",
    project_root / "data" / "dashboard_exports" / "dim_slice.csv",
    project_root / "data" / "dashboard_exports" / "fact_slice_features.csv",
    project_root / "data" / "dashboard_exports" / "fact_anomaly_results.csv",
    project_root / "data" / "dashboard_exports" / "image_index.csv",
    
    project_root / "data" / "sqlite" / "microct_quality_monitoring.db",
    
    project_root / "sql" / "create_tables.sql",
    project_root / "sql" / "example_queries.sql",
    
    project_root / "powerbi" / "microct_quality_monitoring_dashboard.pbix",
    
    project_root / "figures" / "dashboard_01_overview.png",
    project_root / "figures" / "dashboard_02_slice_explorer.png",
]

for path in expected_files:
    print(f"{path.relative_to(project_root)} | exists: {path.exists()}")

README.md | exists: True
requirements.txt | exists: True
notebooks\01_prepare_powerbi_tables.ipynb | exists: True
notebooks\02_export_slice_images_for_dashboard.ipynb | exists: True
notebooks\03_create_sql_database.ipynb | exists: True
data\dashboard_exports\microct_powerbi_input.xlsx | exists: True
data\dashboard_exports\microct_powerbi_input_corrected.xlsx | exists: True
data\dashboard_exports\dim_slice.csv | exists: True
data\dashboard_exports\fact_slice_features.csv | exists: True
data\dashboard_exports\fact_anomaly_results.csv | exists: True
data\dashboard_exports\image_index.csv | exists: True
data\sqlite\microct_quality_monitoring.db | exists: True
sql\create_tables.sql | exists: True
sql\example_queries.sql | exists: True
powerbi\microct_quality_monitoring_dashboard.pbix | exists: True
figures\dashboard_01_overview.png | exists: True
figures\dashboard_02_slice_explorer.png | exists: True


In [16]:
slice_images_dir = project_root / "data" / "dashboard_exports" / "slice_images" / "selected_slices"

png_files = sorted(slice_images_dir.glob("*.png"))

print("Selected slice PNG files:", len(png_files))
print("Expected:", 104)

for file in png_files[:5]:
    print(file.name)

print("...")
for file in png_files[-5:]:
    print(file.name)

Selected slice PNG files: 104
Expected: 104
slice_0000_original.png
slice_0000_pore_mask.png
slice_0000_resin_mask.png
slice_0000_segmentation.png
slice_0001_original.png
...
slice_0415_segmentation.png
slice_0434_original.png
slice_0434_pore_mask.png
slice_0434_resin_mask.png
slice_0434_segmentation.png
